# SofaScore Bundesliga Matchday Odds Extractor

This notebook loads `match_ids_{matchday}.json`, opens every SofaScore odds endpoint through **undetected Chrome 150**, extracts current full-time 1X2 decimal odds, and writes `matchday_{matchday}_odds.json`.

Run the notebook from the directory containing the selected matchday JSON file. The endpoint is opened directly in Chrome as `https://www.sofascore.com/api/v1/event/{match_id}/odds/1/all`; no direct HTTP client is used.

In [ ]:
# Imports
import json
from fractions import Fraction
from pathlib import Path
from pprint import pprint
from typing import Any

# Handle expected failures with a clear, actionable message.
try:
    import undetected_chromedriver as uc
    from selenium.common.exceptions import TimeoutException, WebDriverException
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
except ImportError as exc:
    raise ImportError(
        "Browser dependencies are missing. Install them in this Jupyter kernel with "
        "'%pip install undetected-chromedriver selenium', then restart the kernel."
    ) from exc

In [ ]:
# User parameter
matchday = 1  # Change this value to process another Bundesliga matchday.

# Notebook constants
CHROME_MAJOR_VERSION = 150
# Set workflow configuration value: PAGE_LOAD_TIMEOUT_SECONDS.
PAGE_LOAD_TIMEOUT_SECONDS = 30
# Set workflow configuration value: WAIT_TIMEOUT_SECONDS.
WAIT_TIMEOUT_SECONDS = 20
# Set workflow configuration value: ENDPOINT_TEMPLATE.
ENDPOINT_TEMPLATE = (
    "https://www.sofascore.com/api/v1/event/{match_id}/odds/1/all"
)

In [ ]:
# Validate the parameter and resolve input/output paths.
if not isinstance(matchday, int) or isinstance(matchday, bool) or matchday < 1:
    raise ValueError("matchday must be a positive integer.")

working_directory = Path.cwd()
input_path = working_directory / f"match_ids_{matchday}.json"
output_path = working_directory / f"matchday_{matchday}_odds.json"

print(f"Working directory: {working_directory}")
print(f"Input file:       {input_path}")
print(f"Output file:      {output_path}")

In [ ]:
# Load and validate match_ids_{matchday}.json.
try:
    raw_matches = json.loads(input_path.read_text(encoding="utf-8"))
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f"Input file not found: {input_path}. Place the notebook beside the JSON file."
    ) from exc
except UnicodeDecodeError as exc:
    raise ValueError(f"Input file is not valid UTF-8: {input_path}") from exc
except json.JSONDecodeError as exc:
    raise ValueError(
        f"Input file is not valid JSON (line {exc.lineno}, column {exc.colno}): "
        f"{input_path}"
    ) from exc
except OSError as exc:
    raise OSError(f"Could not read input file {input_path}: {exc}") from exc

# Validate the input before continuing with later processing.
if not isinstance(raw_matches, list):
    raise ValueError("The input JSON must contain a top-level list of matches.")

matches: list[dict[str, Any]] = []
# Process each available item while preserving the current workflow state.
for index, item in enumerate(raw_matches, start=1):
    # Validate the input before continuing with later processing.
    if not isinstance(item, dict):
        raise ValueError(f"Input item {index} must be a JSON object.")

    match_id = item.get("match_id")
    home_team = item.get("home_team")
    away_team = item.get("away_team")

    # Validate the input before continuing with later processing.
    if not isinstance(match_id, int) or isinstance(match_id, bool):
        raise ValueError(f"Input item {index} has no valid integer match_id.")
    # Validate the input before continuing with later processing.
    if not isinstance(home_team, str) or not home_team.strip():
        raise ValueError(f"Input item {index} has no valid home_team.")
    # Validate the input before continuing with later processing.
    if not isinstance(away_team, str) or not away_team.strip():
        raise ValueError(f"Input item {index} has no valid away_team.")

    matches.append(
        {
            "match_id": match_id,
            "home_team": home_team.strip(),
            "away_team": away_team.strip(),
        }
    )

print(f"Loaded and validated {len(matches)} match(es).")

In [ ]:
# Exceptions and fractional-to-decimal conversion.
class OddsNotebookError(RuntimeError):
    pass


# Define Odds Retrieval Error to keep related behaviour explicit.
class OddsRetrievalError(OddsNotebookError):
    pass


# Define Odds Not Found Error to keep related behaviour explicit.
class OddsNotFoundError(OddsNotebookError):
    pass


# Define Invalid Odds JSONError to keep related behaviour explicit.
class InvalidOddsJSONError(OddsNotebookError):
    pass


# Define Odds Unavailable Error to keep related behaviour explicit.
class OddsUnavailableError(OddsNotebookError):
    pass


# Handle to decimal for reuse in the workflow.
def fractional_to_decimal(value: Any) -> float:
    # Validate the input before continuing with later processing.
    if not isinstance(value, str) or not value.strip():
        raise ValueError("fractionalValue must be a non-empty string.")

    # Handle expected failures with a clear, actionable message.
    try:
        fractional_odds = Fraction(value.strip())
    except (ValueError, ZeroDivisionError) as exc:
        raise ValueError(f"Invalid fractional odds value: {value!r}.") from exc

    # Validate the input before continuing with later processing.
    if fractional_odds < 0:
        raise ValueError(f"Fractional odds cannot be negative: {value!r}.")

    decimal_odds = float(fractional_odds + 1)
    # Validate the input before continuing with later processing.
    if decimal_odds < 1:
        raise ValueError(f"Invalid decimal odds converted from {value!r}.")
    return round(decimal_odds, 4)

In [ ]:
# Configure undetected Chrome 150.
options = uc.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.set_capability("goog:loggingPrefs", {"performance": "ALL"})

In [ ]:
# Initialize one reusable undetected Chrome 150 driver.
driver = None
# Handle expected failures with a clear, actionable message.
try:
    driver = uc.Chrome(
        options=options,
        version_main=CHROME_MAJOR_VERSION,
        use_subprocess=True,
    )
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT_SECONDS)
    driver.execute_cdp_cmd("Network.enable", {})
except Exception as exc:
    if driver is not None:
        # Handle expected failures with a clear, actionable message.
        try:
            driver.quit()
        except Exception:
            pass
        driver = None
    raise RuntimeError(
        "Could not initialize undetected Chrome 150. Confirm that a Chrome "
        f"150-compatible installation is available. Original error: {exc}"
    ) from exc

print("Undetected Chrome 150 initialized.")

In [ ]:
# Helpers for reading the HTTP status of Chrome's document navigation.
def clear_performance_log(driver_instance: Any) -> None:
    # Handle expected failures with a clear, actionable message.
    try:
        driver_instance.get_log("performance")
    except Exception:
        # Status detection is optional; the response body remains the fallback.
        pass


# Read performance log for reuse in the workflow.
def read_performance_log(driver_instance: Any) -> list[dict[str, Any]]:
    # Handle expected failures with a clear, actionable message.
    try:
        return driver_instance.get_log("performance")
    except Exception:
        return []


# Find document status for reuse in the workflow.
def find_document_status(
    log_entries: list[dict[str, Any]],
    requested_url: str,
    current_url: str | None = None,
) -> int | None:
    candidate_urls = {requested_url.rstrip("/")}
    if current_url:
        candidate_urls.add(current_url.rstrip("/"))

    observed_status: int | None = None
    # Process each available item while preserving the current workflow state.
    for entry in log_entries:
        # Handle expected failures with a clear, actionable message.
        try:
            message = json.loads(entry["message"])["message"]
            if message.get("method") != "Network.responseReceived":
                continue

            params = message.get("params", {})
            response = params.get("response", {})
            response_url = str(response.get("url", "")).rstrip("/")
            if params.get("type") != "Document":
                continue
            if response_url not in candidate_urls:
                continue

            observed_status = int(float(response["status"]))
        except (KeyError, TypeError, ValueError, json.JSONDecodeError):
            continue

    return observed_status

In [ ]:
# Retrieve and parse the SofaScore odds JSON for one match.
def payload_reports_404(payload: dict[str, Any]) -> bool:
    candidates: list[Any] = [
        payload.get("status"),
        payload.get("statusCode"),
        payload.get("code"),
    ]
    error_value = payload.get("error")
    # Choose the appropriate path for the current data state.
    if isinstance(error_value, dict):
        candidates.extend(
            [
                error_value.get("status"),
                error_value.get("statusCode"),
                error_value.get("code"),
            ]
        )
    elif isinstance(error_value, str) and "404" in error_value:
        return True

    return any(str(value).strip() == "404" for value in candidates if value is not None)


# Handle odds json for reuse in the workflow.
def retrieve_odds_json(match_id: int) -> dict[str, Any]:
    url = (
        f"https://www.sofascore.com/api/v1/"
        f"event/{match_id}/odds/1/all"
    )

    clear_performance_log(driver)
    # Handle expected failures with a clear, actionable message.
    try:
        driver.get(url)
    except TimeoutException as exc:
        log_entries = read_performance_log(driver)
        status = find_document_status(log_entries, url, driver.current_url)
        # Validate the input before continuing with later processing.
        if status == 404:
            raise OddsNotFoundError(
                "SofaScore returned HTTP 404; no odds are available for this match."
            ) from exc
        raise OddsRetrievalError(
            f"Timed out after {PAGE_LOAD_TIMEOUT_SECONDS} seconds loading {url}."
        ) from exc
    except WebDriverException as exc:
        raise OddsRetrievalError(f"Chrome could not load {url}: {exc}") from exc

    log_entries = read_performance_log(driver)
    status = find_document_status(log_entries, url, driver.current_url)
    # Validate the input before continuing with later processing.
    if status == 404:
        raise OddsNotFoundError(
            "SofaScore returned HTTP 404; no odds are available for this match."
        )
    # Validate the input before continuing with later processing.
    if status is not None and status >= 400:
        raise OddsRetrievalError(f"SofaScore returned HTTP {status} for {url}.")

    # Handle empty body text for reuse in the workflow.
    def non_empty_body_text(current_driver: Any) -> str | bool:
        body_text = current_driver.find_element(By.TAG_NAME, "body").text.strip()
        return body_text if body_text else False

    # Handle expected failures with a clear, actionable message.
    try:
        response_text = WebDriverWait(driver, WAIT_TIMEOUT_SECONDS).until(
            non_empty_body_text
        )
    except TimeoutException as exc:
        raise OddsRetrievalError(
            f"SofaScore returned no readable body within {WAIT_TIMEOUT_SECONDS} seconds."
        ) from exc
    except WebDriverException as exc:
        raise OddsRetrievalError(
            f"Chrome could not read the SofaScore response body: {exc}"
        ) from exc

    # Handle expected failures with a clear, actionable message.
    try:
        payload = json.loads(response_text)
    except json.JSONDecodeError as exc:
        raise InvalidOddsJSONError(
            "SofaScore did not return valid JSON "
            f"(line {exc.lineno}, column {exc.colno})."
        ) from exc

    # Validate the input before continuing with later processing.
    if not isinstance(payload, dict):
        raise InvalidOddsJSONError("The SofaScore response must be a JSON object.")
    # Validate the input before continuing with later processing.
    if payload_reports_404(payload):
        raise OddsNotFoundError(
            "SofaScore returned HTTP 404; no odds are available for this match."
        )

    return payload

In [ ]:
# Extract current full-time 1X2 decimal odds.
def extract_full_time_1x2(
    payload: dict[str, Any],
) -> tuple[list[dict[str, Any]], list[str]]:
    markets = payload.get("markets")
    # Validate the input before continuing with later processing.
    if not isinstance(markets, list):
        raise OddsUnavailableError("The response does not contain a markets list.")

    bookmakers: list[dict[str, Any]] = []
    issues: list[str] = []
    qualifying_market_count = 0
    required_outcomes = {"1", "X", "2"}

    # Process each available item while preserving the current workflow state.
    for market_index, market in enumerate(markets, start=1):
        if not isinstance(market, dict):
            continue
        if not (
            market.get("marketId") == 1
            and market.get("marketGroup") == "1X2"
            and market.get("marketPeriod") == "Full-time"
        ):
            continue

        qualifying_market_count += 1
        source_id = market.get("sourceId")
        market_label = f"market {market_index} (sourceId={source_id!r})"

        if market.get("suspended") is True:
            issues.append(f"Skipped suspended {market_label}.")
            continue
        if source_id is None:
            issues.append(f"Skipped {market_label}: sourceId is missing.")
            continue

        choices = market.get("choices")
        if not isinstance(choices, list):
            issues.append(f"Skipped {market_label}: choices is not a list.")
            continue

        outcomes: dict[str, dict[str, Any]] = {}
        duplicate_outcomes: set[str] = set()
        # Process each available item while preserving the current workflow state.
        for choice in choices:
            if not isinstance(choice, dict):
                continue
            name = choice.get("name")
            if name not in required_outcomes:
                continue
            # Choose the appropriate path for the current data state.
            if name in outcomes:
                duplicate_outcomes.add(name)
            else:
                outcomes[name] = choice

        missing_outcomes = required_outcomes - outcomes.keys()
        if missing_outcomes or duplicate_outcomes:
            details: list[str] = []
            if missing_outcomes:
                details.append("missing " + ", ".join(sorted(missing_outcomes)))
            if duplicate_outcomes:
                details.append("duplicate " + ", ".join(sorted(duplicate_outcomes)))
            issues.append(f"Skipped {market_label}: {'; '.join(details)} outcome(s).")
            continue

        # Handle expected failures with a clear, actionable message.
        try:
            home_win = fractional_to_decimal(outcomes["1"].get("fractionalValue"))
            draw = fractional_to_decimal(outcomes["X"].get("fractionalValue"))
            away_win = fractional_to_decimal(outcomes["2"].get("fractionalValue"))
        except ValueError as exc:
            issues.append(f"Skipped {market_label}: {exc}")
            continue

        bookmakers.append(
            {
                "bookmaker": None,
                "bookmaker_source_id": source_id,
                "home_win": home_win,
                "draw": draw,
                "away_win": away_win,
            }
        )

    # Validate the input before continuing with later processing.
    if qualifying_market_count == 0:
        raise OddsUnavailableError("No full-time 1X2 market is available.")
    if not bookmakers and not issues:
        issues.append("No valid full-time 1X2 odds could be extracted.")

    return bookmakers, issues

In [ ]:
# Process every match independently so one failure does not stop the matchday.
results: list[dict[str, Any]] = []
total_matches = len(matches)

# Process each available item while preserving the current workflow state.
for match_number, match in enumerate(matches, start=1):
    match_id = match["match_id"]
    home_team = match["home_team"]
    away_team = match["away_team"]
    result: dict[str, Any] = {
        "match_id": match_id,
        "home_team": home_team,
        "away_team": away_team,
        "bookmakers": [],
        "issues": [],
    }

    print(
        f"[{match_number}/{total_matches}] {home_team} vs {away_team} "
        f"(match_id={match_id})"
    )

    # Handle expected failures with a clear, actionable message.
    try:
        payload = retrieve_odds_json(match_id)
        bookmakers, extraction_issues = extract_full_time_1x2(payload)
        result["bookmakers"] = bookmakers
        result["issues"].extend(extraction_issues)
        # Choose the appropriate path for the current data state.
        if bookmakers:
            print(f"  Extracted {len(bookmakers)} source/bookmaker record(s).")
        else:
            print("  No valid odds extracted.")
    except OddsNotFoundError as exc:
        result["issues"].append(str(exc))
        print(f"  404: {exc}")
    except OddsNotebookError as exc:
        result["issues"].append(f"{type(exc).__name__}: {exc}")
        print(f"  Issue: {type(exc).__name__}: {exc}")
    except Exception as exc:
        result["issues"].append(f"Unexpected error: {type(exc).__name__}: {exc}")
        print(f"  Unexpected error: {type(exc).__name__}: {exc}")

    results.append(result)

print("Processing complete.")

In [ ]:
# Inspect the collected result and a concise summary.
pprint(results, sort_dicts=False)

matches_with_odds = sum(bool(result["bookmakers"]) for result in results)
matches_without_odds = len(results) - matches_with_odds
matches_with_issues = sum(bool(result["issues"]) for result in results)
bookmaker_records = sum(len(result["bookmakers"]) for result in results)

print("\nSummary")
print(f"  Total input matches:       {len(matches)}")
print(f"  Matches with odds:         {matches_with_odds}")
print(f"  Matches without odds:      {matches_without_odds}")
print(f"  Matches with issues:       {matches_with_issues}")
print(f"  Source/bookmaker records:  {bookmaker_records}")

In [ ]:
# Save matchday_{matchday}_odds.json as readable UTF-8 JSON.
try:
    output_path.write_text(
        json.dumps(results, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
except OSError as exc:
    raise OSError(f"Could not save {output_path}: {exc}") from exc

print(f"Saved {len(results)} match result(s) to {output_path.resolve()}.")

In [ ]:
# Close the Chrome driver after saving the results.
if driver is not None:
    # Handle expected failures with a clear, actionable message.
    try:
        driver.quit()
        print("Chrome driver closed.")
    except Exception as exc:
        print(f"Chrome driver shutdown warning: {exc}")
    finally:
        driver = None
else:
    print("Chrome driver is already closed or was not initialized.")